In [ ]:
# @title
# -*- coding: utf-8 -*-
"""
One-cell Colab runner: fair seed benchmark, Gemma 4 26B Q3_K_M vs E4B BF16.

Copy this whole file into one Colab Python cell and run it. No environment
variables are needed for the model pair.

This benchmark is deliberately harder and fairer than testing on the DFUZZ
training JSONL directly:

- Builds the test set from the PyTorch version currently installed in the
  runtime, not from stored reference outputs.
- Excludes API names seen in the old DFUZZ seed dataset when possible.
- Uses a neutral English prompt, not the original Vietnamese training prompt.
- Gives both models the exact same API list, prompt template, sampling config,
  max tokens, timeout, and validation rules.
- Reports pass rate plus generation cost: time, raw token count, and code lines.

Fixed model order:
  1. Public Gemma 4 26B-A4B-it Q3_K_M baseline.
  2. Public Gemma 4 E4B-it BF16, not quantized.
"""

import ast
import gc
import importlib
import importlib.util
import inspect
import json
import os
import random
import re
import subprocess
import sys
import tempfile
import time
from datetime import datetime
from pathlib import Path


# =============================================================================
# CONFIG
# =============================================================================

MODEL_SPECS = [
    {
        "label": "base_gemma4_26b_q3km",
        "repo_id": "mradermacher/gemma-4-26B-A4B-it-GGUF",
        "filename": "gemma-4-26B-A4B-it.Q3_K_M.gguf",
    },
    {
        "label": "base_gemma4_e4b_bf16",
        "repo_id": "unsloth/gemma-4-E4B-it-GGUF",
        "filename": "gemma-4-E4B-it-BF16.gguf",
    },
]

N_CTX = int(os.environ.get("N_CTX", "4096"))
N_GPU_LAYERS = int(os.environ.get("N_GPU_LAYERS", "-1"))
REQUESTED_N_CTX = N_CTX
REQUESTED_N_GPU_LAYERS = N_GPU_LAYERS

LIMIT_TESTS = int(os.environ.get("LIMIT_TESTS", "40"))
MAX_ATTEMPTS = int(os.environ.get("MAX_ATTEMPTS", "2"))
TIMEOUT_S = float(os.environ.get("TIMEOUT_S", "10.0"))
SEED = int(os.environ.get("SEED", "3407"))

FORCE_CPU_TESTS = os.environ.get("FORCE_CPU_TESTS", "1") == "1"
VERBOSE_GEN = os.environ.get("VERBOSE_GEN", "1") == "1"
SHOW_CODE_TABLES = os.environ.get("SHOW_CODE_TABLES", "0") == "1"

TEMPERATURES = [float(x) for x in os.environ.get("TEMPERATURES", "0.2,0.35").split(",")]
TOP_P = float(os.environ.get("TOP_P", "0.95"))
TOP_K = int(os.environ.get("TOP_K", "40"))
MIN_P = float(os.environ.get("MIN_P", "0.05"))
REPEAT_PENALTY = float(os.environ.get("REPEAT_PENALTY", "1.08"))
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "600"))

# current_ood: prefer APIs not present in the DFUZZ seed dataset.
# current_all: sample all current-runtime APIs.
EVAL_MODE = os.environ.get("EVAL_MODE", "current_ood")
API_POOL = os.environ.get("API_POOL", "mixed")  # mixed | functions | tensor_methods

TRAIN_DATASET_PATH = Path(os.environ.get("TRAIN_DATASET_PATH", "fuzz_finetune_dataset_REAL.jsonl"))
REPORT_FILE = os.environ.get("REPORT_FILE", "benchmark_seed_generator_base26_vs_e4b_bf16.json")
DEBUG_REPORT_FILE = os.environ.get("DEBUG_REPORT_FILE", "benchmark_seed_generator_base26_vs_e4b_bf16_debug.json")

RANDOM = random.Random(SEED)


# =============================================================================
# DEPENDENCIES
# =============================================================================

def _pip_install(packages, extra_args=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages]
    if extra_args:
        cmd.extend(extra_args)
    subprocess.check_call(cmd)
    importlib.invalidate_caches()


def ensure_deps():
    missing = []
    for mod, pkg in [("huggingface_hub", "huggingface_hub"), ("rich", "rich")]:
        if importlib.util.find_spec(mod) is None:
            missing.append(pkg)
    if missing:
        print("Installing:", " ".join(missing))
        _pip_install(missing)

    if importlib.util.find_spec("llama_cpp") is None:
        print("Installing llama-cpp-python CUDA wheel...")
        _pip_install(
            ["llama-cpp-python"],
            ["--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"],
        )


ensure_deps()

import torch
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table

console = Console()


def ts():
    return datetime.now().strftime("%H:%M:%S")


def log(tag, msg, style=""):
    if style:
        console.print(f"[dim]{ts()}[/dim] {tag} [{style}]{msg}[/{style}]")
    else:
        console.print(f"[dim]{ts()}[/dim] {tag} {msg}")


# =============================================================================
# API SET FROM CURRENT PYTORCH
# =============================================================================

SAFE_SKIP_PATTERNS = re.compile(
    r"(distributed|elastic|multiprocessing|rendezvous|rpc|cuda|mps|xpu|"
    r"hub\.download|download_url|save|load|serialization|cpp_extension|"
    r"Storage|UntypedStorage|TypedStorage|set_|fork|spawn|process|profiler|"
    r"compile|jit|export|package|deploy|benchmark|autograd\.profiler)",
    re.IGNORECASE,
)


def safe_doc(obj, limit=2200):
    try:
        doc = inspect.getdoc(obj) or ""
    except Exception:
        doc = ""
    doc = re.sub(r"\n{3,}", "\n\n", doc).strip()
    if len(doc) > limit:
        doc = doc[: limit // 2] + "\n... [doc truncated] ...\n" + doc[-limit // 2 :]
    return doc


def safe_signature(obj):
    try:
        return str(inspect.signature(obj))
    except Exception:
        text = safe_doc(obj, 500)
        first = text.splitlines()[0] if text else ""
        return first[:220] if first else "(signature unavailable)"


def is_reasonable_callable(api_name, obj):
    if SAFE_SKIP_PATTERNS.search(api_name):
        return False
    if api_name.startswith("torch._") or ".__" in api_name:
        return False
    short_name = api_name.rsplit(".", 1)[-1]
    if short_name.startswith("_") or short_name in {"typename", "type"}:
        return False
    if inspect.ismodule(obj):
        return False
    if not callable(obj):
        return False
    return True


def collect_current_torch_apis():
    entries = []

    module_specs = [
        ("torch", torch),
        ("torch.nn.functional", torch.nn.functional),
        ("torch.linalg", torch.linalg),
        ("torch.special", torch.special),
        ("torch.fft", torch.fft),
    ]

    for prefix, module in module_specs:
        for name in dir(module):
            if name.startswith("_"):
                continue
            api = f"{prefix}.{name}"
            try:
                obj = getattr(module, name)
            except Exception:
                continue
            if not is_reasonable_callable(api, obj):
                continue
            entries.append({
                "api": api,
                "kind": "function_or_class",
                "signature": safe_signature(obj),
                "doc": safe_doc(obj),
            })

    for name in dir(torch.Tensor):
        if name.startswith("_"):
            continue
        api = f"torch.Tensor.{name}"
        try:
            obj = getattr(torch.Tensor, name)
        except Exception:
            continue
        if not is_reasonable_callable(api, obj):
            continue
        entries.append({
            "api": api,
            "kind": "tensor_method",
            "signature": safe_signature(obj),
            "doc": safe_doc(obj),
        })

    # De-duplicate by API name while keeping deterministic order.
    seen = set()
    unique = []
    for entry in sorted(entries, key=lambda x: x["api"]):
        if entry["api"] in seen:
            continue
        seen.add(entry["api"])
        unique.append(entry)
    return unique


API_LINE_PREFIXES = (
    "Muc tieu API:",
    "API Target:",
    "Target API:",
    "\u004d\u1ee5c ti\u00eau API:",
)


def api_bucket(api):
    if api.startswith("torch.Tensor."):
        return "torch.Tensor"
    if api.startswith("torch.nn.functional."):
        return "torch.nn.functional"
    if api.startswith("torch.linalg."):
        return "torch.linalg"
    if api.startswith("torch.special."):
        return "torch.special"
    if api.startswith("torch.fft."):
        return "torch.fft"
    return "torch"


def balanced_sample(entries, limit):
    buckets = {}
    for entry in entries:
        buckets.setdefault(api_bucket(entry["api"]), []).append(entry)
    for bucket_entries in buckets.values():
        RANDOM.shuffle(bucket_entries)

    selected = []
    bucket_names = sorted(buckets)
    while len(selected) < limit and any(buckets.values()):
        for name in bucket_names:
            if buckets[name]:
                selected.append(buckets[name].pop())
                if len(selected) >= limit:
                    break
    return selected


def find_train_dataset():
    candidates = [
        TRAIN_DATASET_PATH,
        Path.cwd() / TRAIN_DATASET_PATH,
        Path("/content") / TRAIN_DATASET_PATH.name,
        Path.home() / TRAIN_DATASET_PATH.name,
        Path(r"C:\Users\fagon\OneDrive\Desktop\New folder\fuzz_finetune_dataset_REAL.jsonl"),
    ]
    for p in candidates:
        if p.exists() and p.is_file():
            return p
    return None


def extract_api_from_instruction(instruction):
    for line in instruction.splitlines():
        stripped = line.strip()
        for prefix in API_LINE_PREFIXES:
            if stripped.startswith(prefix):
                target = stripped[len(prefix):].strip()
                if target:
                    return target
    match = re.search(r"\btorch(?:\.[A-Za-z_]\w*)+\b", instruction)
    return match.group(0) if match else ""


def load_seen_train_apis():
    path = find_train_dataset()
    seen = set()
    if not path:
        return seen, None
    try:
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                item = json.loads(line)
                api = extract_api_from_instruction(item.get("instruction", ""))
                if api:
                    seen.add(api)
    except Exception as exc:
        log("data", f"Could not parse train dataset for API exclusion: {exc}", "yellow")
    return seen, str(path)


def select_eval_apis():
    all_entries = collect_current_torch_apis()
    seen_train, train_path = load_seen_train_apis()

    if API_POOL == "functions":
        all_entries = [x for x in all_entries if x["kind"] == "function_or_class"]
    elif API_POOL == "tensor_methods":
        all_entries = [x for x in all_entries if x["kind"] == "tensor_method"]

    ood_entries = [x for x in all_entries if x["api"] not in seen_train]
    if EVAL_MODE == "current_ood":
        pool = ood_entries
        if len(pool) < LIMIT_TESTS:
            log("data", f"Only {len(pool)} OOD APIs available; filling from current_all.", "yellow")
            used = {x["api"] for x in pool}
            pool = pool + [x for x in all_entries if x["api"] not in used]
    elif EVAL_MODE == "current_all":
        pool = all_entries
    else:
        raise ValueError(f"Unknown EVAL_MODE={EVAL_MODE!r}")

    selected = balanced_sample(pool, LIMIT_TESTS)
    selected_bucket_counts = {}
    for entry in selected:
        bucket = api_bucket(entry["api"])
        selected_bucket_counts[bucket] = selected_bucket_counts.get(bucket, 0) + 1
    metadata = {
        "torch_version": torch.__version__,
        "torch_cuda": torch.version.cuda,
        "eval_mode": EVAL_MODE,
        "api_pool": API_POOL,
        "all_current_api_count": len(all_entries),
        "train_dataset_path": train_path,
        "seen_train_api_count": len(seen_train),
        "ood_current_api_count": len(ood_entries),
        "selected_api_count": len(selected),
        "selected_bucket_counts": selected_bucket_counts,
        "selected_apis": [x["api"] for x in selected],
    }
    return selected, metadata


# =============================================================================
# MODEL + GENERATION
# =============================================================================

llm = None
_has_builtin_template = False


def load_llm_with_fallback(path):
    global N_CTX, N_GPU_LAYERS
    requested_ctx = N_CTX
    requested_layers = N_GPU_LAYERS
    candidates = [(requested_ctx, requested_layers)]

    if requested_layers == -1:
        layer_fallbacks = [64, 56, 48, 40, 32, 24]
    elif requested_layers > 24:
        layer_fallbacks = [requested_layers, 48, 40, 32, 24]
    else:
        layer_fallbacks = [requested_layers]

    for ctx in [requested_ctx, 3072, 2048]:
        for layers in layer_fallbacks:
            candidates.append((ctx, layers))

    seen = set()
    errors = []
    for ctx, layers in candidates:
        if (ctx, layers) in seen:
            continue
        seen.add((ctx, layers))
        try:
            log("model", f"Trying GGUF load: n_ctx={ctx}, n_gpu_layers={layers}", "yellow")
            model = Llama(model_path=path, n_gpu_layers=layers, n_ctx=ctx, verbose=False)
            N_CTX = ctx
            N_GPU_LAYERS = layers
            log("model", f"Loaded with n_ctx={N_CTX}, n_gpu_layers={N_GPU_LAYERS}", "green")
            return model
        except Exception as exc:
            errors.append(f"n_ctx={ctx}, n_gpu_layers={layers}: {exc}")
            log("model", f"Load failed at n_ctx={ctx}, layers={layers}: {exc}", "red")
            gc.collect()
    raise RuntimeError("Could not load GGUF on this runtime:\n" + "\n".join(errors[-8:]))


def load_model(spec):
    global llm, _has_builtin_template, N_CTX, N_GPU_LAYERS
    N_CTX = REQUESTED_N_CTX
    N_GPU_LAYERS = REQUESTED_N_GPU_LAYERS
    log("model", f"Downloading/loading {spec['repo_id']} / {spec['filename']} ...", "yellow")
    model_path = hf_hub_download(repo_id=spec["repo_id"], filename=spec["filename"])
    llm = load_llm_with_fallback(model_path)

    _has_builtin_template = False
    try:
        llm.create_chat_completion(messages=[{"role": "user", "content": "test"}], max_tokens=1)
        _has_builtin_template = True
        log("model", "GGUF chat template works; using create_chat_completion().", "green")
    except Exception as exc:
        log("model", f"No usable embedded chat template: {exc}", "yellow")
        log("model", "Using manual Gemma turn formatting.", "yellow")


def unload_model():
    global llm
    try:
        del llm
    except Exception:
        pass
    llm = None
    gc.collect()


def manual_gemma_prompt(messages):
    parts = []
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            parts.append(f"<bos><start_of_turn>system\n{content}<end_of_turn>")
        elif role == "user":
            parts.append(f"<start_of_turn>user\n{content}<end_of_turn>")
        elif role == "assistant":
            parts.append(f"<start_of_turn>model\n{content}<end_of_turn>")
    parts.append("<start_of_turn>model\n")
    return "\n".join(parts)


def stream_llm_response(messages, temperature=0.2, max_tokens=MAX_TOKENS, top_p=TOP_P):
    pieces = []
    repetition_triggered = False
    stop = ["<end_of_turn>", "<start_of_turn>"]
    sampling = {
        "temperature": temperature,
        "top_p": top_p,
        "top_k": TOP_K,
        "min_p": MIN_P,
        "repeat_penalty": REPEAT_PENALTY,
        "max_tokens": max_tokens,
        "stop": stop,
        "chat_template": "embedded" if _has_builtin_template else "manual_gemma",
    }

    if VERBOSE_GEN:
        console.print(
            f"\n[bold cyan]--- RAW MODEL OUTPUT "
            f"temp={temperature} top_p={top_p} top_k={TOP_K} min_p={MIN_P} "
            f"repeat_penalty={REPEAT_PENALTY} max_tokens={max_tokens} ---[/bold cyan]"
        )

    t0 = time.time()
    if _has_builtin_template:
        stream = llm.create_chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=TOP_K,
            min_p=MIN_P,
            repeat_penalty=REPEAT_PENALTY,
            stop=stop,
            stream=True,
        )
        for chunk in stream:
            delta = chunk["choices"][0].get("delta", {})
            content = delta.get("content")
            if not content:
                continue
            pieces.append(content)
            if VERBOSE_GEN:
                sys.stdout.write(content)
                sys.stdout.flush()
            if len(pieces) % 20 == 0:
                recent = [l.strip() for l in "".join(pieces).splitlines()[-20:] if len(l.strip()) > 5]
                if any(recent.count(line) >= 7 for line in recent):
                    repetition_triggered = True
                    break
    else:
        stream = llm(
            manual_gemma_prompt(messages),
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=TOP_K,
            min_p=MIN_P,
            repeat_penalty=REPEAT_PENALTY,
            stop=stop,
            stream=True,
            echo=False,
        )
        for chunk in stream:
            content = chunk["choices"][0].get("text", "")
            pieces.append(content)
            if VERBOSE_GEN:
                sys.stdout.write(content)
                sys.stdout.flush()
            if len(pieces) % 20 == 0:
                recent = [l.strip() for l in "".join(pieces).splitlines()[-20:] if len(l.strip()) > 5]
                if any(recent.count(line) >= 7 for line in recent):
                    repetition_triggered = True
                    break

    raw_text = "".join(pieces)
    token_ids = []
    try:
        token_ids = llm.tokenize(raw_text.encode("utf-8"), add_bos=False)
    except Exception:
        pass

    if VERBOSE_GEN:
        print()
        console.print("[bold cyan]--- END RAW MODEL OUTPUT ---[/bold cyan]\n")

    raw_record = {
        "raw_text": raw_text,
        "raw_chunks": pieces,
        "raw_token_ids": token_ids,
        "raw_token_count": len(token_ids) if token_ids else None,
        "generation_time_s": round(time.time() - t0, 3),
        "sampling": sampling,
    }
    return raw_text, repetition_triggered, raw_record


# =============================================================================
# PROMPT + VALIDATION
# =============================================================================

FAIR_SYSTEM_PROMPT = (
    "You are a PyTorch seed harness generator. "
    "Output only one complete Python program. No markdown, no explanation."
)


def build_seed_messages(entry):
    api = entry["api"]
    return [
        {"role": "system", "content": FAIR_SYSTEM_PROMPT},
        {"role": "user", "content": f"""Current runtime:
- PyTorch version: {torch.__version__}
- CUDA version reported by torch: {torch.version.cuda}
- Evaluation subprocess uses CPU unless FORCE_CPU_TESTS=0.

Target API: {api}
API kind: {entry['kind']}
Signature or first doc line: {entry['signature']}

Documentation excerpt:
{entry['doc'] or '(no docstring available)'}

Task:
Create a minimal executable seed/harness program for the target API on tiny deterministic inputs.

Rules:
- Start with import torch.
- Actually call the exact target API once or more.
- Use tiny tensors or tiny Python values only.
- Materialize the result with print(...) or a simple variable that is printed.
- Do not use network, files, subprocess, ctypes, multiprocessing, or huge allocations.
- Do not wrap the target call in try/except. Let real errors surface.
- Output only Python code, not a fenced markdown block.
"""},
    ]


def extract_python_code(text):
    if not text:
        return ""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<end_of_turn>", "").replace("end_of_turn>", "")
    blocks = re.findall(r"```(?:python|py)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if blocks:
        code = blocks[-1].strip()
    else:
        start = text.find("import torch")
        if start < 0:
            start = text.find("from torch")
        code = text[start:].strip() if start >= 0 else text.strip()
    bad_prefixes = ("Here is", "Sure", "Explanation", "Analysis", "```")
    return "\n".join(line.rstrip() for line in code.splitlines() if not line.strip().startswith(bad_prefixes)).strip()


def target_markers(api_target):
    parts = api_target.split(".")
    last = parts[-1] if parts else api_target
    markers = {f"{api_target}("}
    if api_target.startswith("torch.nn.functional."):
        markers.add(f"F.{last}(")
        markers.add(f"functional.{last}(")
        markers.add(f"nn.functional.{last}(")
    if api_target.startswith("torch.Tensor."):
        markers.add(f".{last}(")
        markers.add(f"torch.Tensor.{last}(")
    if api_target.startswith("torch.linalg."):
        markers.add(f"linalg.{last}(")
    if api_target.startswith("torch.special."):
        markers.add(f"special.{last}(")
    if api_target.startswith("torch.fft."):
        markers.add(f"fft.{last}(")
    return markers


BANNED_PATTERNS = [
    (r"\bctypes\b", "ctypes is not allowed"),
    (r"\bsubprocess\b", "subprocess is not allowed"),
    (r"\bos\.system\b", "shell execution is not allowed"),
    (r"\bsocket\b|\brequests\b|\burllib\b", "network access is not allowed"),
    (r"\bopen\s*\(|\bPath\s*\(", "file I/O is not allowed"),
    (r"\bshutil\.rmtree\b|\bos\.remove\b|\bos\.unlink\b", "destructive filesystem calls are not allowed"),
    (r"\bsys\.exit\b|\bexit\s*\(|\bquit\s*\(", "self-exit hides seed quality"),
    (r"\braise\s+(RuntimeError|Exception|AssertionError|BaseException)", "manual raise hides seed quality"),
]


def validate_seed_code(code, api_target):
    issues = []
    warnings = []
    if not code:
        issues.append("no code")
        return issues, warnings
    if "import torch" not in code and "from torch" not in code:
        issues.append("missing import torch")
    try:
        tree = ast.parse(code)
    except SyntaxError as exc:
        issues.append(f"syntax error: {exc}")
        return issues, warnings

    compact = code.replace(" ", "")
    if not any(marker.replace(" ", "") in compact for marker in target_markers(api_target)):
        issues.append(f"does not appear to call target API {api_target}")

    for pattern, reason in BANNED_PATTERNS:
        if re.search(pattern, code):
            issues.append(reason)

    for node in ast.walk(tree):
        if isinstance(node, ast.Try):
            warnings.append("uses try/except; may hide API failure")
        if isinstance(node, (ast.For, ast.While, ast.AsyncFor)):
            warnings.append("uses loop; may be slower than necessary")

    if "print(" not in code:
        warnings.append("does not print or expose result")
    if len([x for x in code.splitlines() if x.strip()]) > 45:
        warnings.append("seed is verbose (>45 non-empty lines)")
    if re.search(r"\bassert\b", code):
        warnings.append("uses assert/unit-test style")
    return issues, sorted(set(warnings))


def run_code_file(code, temp_dir, name):
    path = Path(temp_dir) / name
    path.write_text(code, encoding="utf-8")
    env = os.environ.copy()
    env.setdefault("PYTHONFAULTHANDLER", "1")
    env.setdefault("OMP_NUM_THREADS", "1")
    env.setdefault("MKL_NUM_THREADS", "1")
    if FORCE_CPU_TESTS:
        env["CUDA_VISIBLE_DEVICES"] = ""
    t0 = time.time()
    proc = subprocess.run(
        [sys.executable, str(path)],
        capture_output=True,
        text=True,
        timeout=TIMEOUT_S,
        env=env,
    )
    return proc, round(time.time() - t0, 3)


def feedback_for_seed(status, detail, api_target):
    if status == "NO_CODE":
        return "Output only a Python program. Start with import torch. No markdown."
    if status == "SYNTAX_ERROR":
        return f"Syntax failed: {detail}. Repair it and output only Python code."
    if status == "TARGET_MISS":
        return f"The program did not call {api_target}. Regenerate a tiny executable seed that calls this exact API."
    if status == "FORBIDDEN":
        return f"Forbidden constructs: {detail}. Use only torch/tiny values and a direct target API call."
    if status == "RUNTIME_ERROR":
        return f"The seed crashed instead of exiting 0:\n{detail[:700]}\nRepair it."
    if status == "TIMEOUT":
        return "The seed timed out. Make it direct and tiny."
    return "Regenerate a smaller valid seed program that calls the target API and exits 0."


def evaluate_status(issues, proc):
    if issues:
        joined = "; ".join(issues)
        if "no code" in joined:
            return "NO_CODE", joined
        if "syntax error" in joined:
            return "SYNTAX_ERROR", joined
        if "does not appear" in joined:
            return "TARGET_MISS", joined
        return "FORBIDDEN", joined
    if proc is None:
        return "NOT_RUN", ""
    if proc.returncode != 0:
        err = (proc.stderr or proc.stdout or "").strip()
        return "RUNTIME_ERROR", err.splitlines()[-1] if err else f"returncode {proc.returncode}"
    return "PASS_RUNNABLE_SEED", "exit 0"


# =============================================================================
# BENCHMARK
# =============================================================================

def empty_results(spec):
    return {
        "label": spec["label"],
        "model": f"{spec['repo_id']} / {spec['filename']}",
        "task": "Fair current-runtime PyTorch seed generation",
        "n_ctx": N_CTX,
        "n_gpu_layers": N_GPU_LAYERS,
        "limit_tests": LIMIT_TESTS,
        "max_attempts": MAX_ATTEMPTS,
        "interrupted": False,
        "total_attempted": 0,
        "pass_runnable_seed": 0,
        "pass_oneshot": 0,
        "fail_no_code": 0,
        "fail_syntax": 0,
        "fail_target_miss": 0,
        "fail_forbidden": 0,
        "fail_runtime": 0,
        "fail_timeout": 0,
        "fail_repetition": 0,
        "fail_context_overflow": 0,
        "total_generation_time_s": 0.0,
        "total_subprocess_time_s": 0.0,
        "raw_token_counts": [],
        "code_line_counts": [],
        "warning_counts": {},
        "details": [],
    }


def count_code_lines(code):
    return len([line for line in code.splitlines() if line.strip()])


def run_one_seed_task(spec, entry, idx, total_records, temp_dir):
    t0 = time.time()
    api_target = entry["api"]
    messages = build_seed_messages(entry)
    final_code = ""
    final_status = "FAILED"
    final_detail = ""
    final_warnings = []
    attempt_debugs = []
    pass_oneshot = False
    total_gen_time = 0.0
    total_proc_time = 0.0
    final_token_count = None

    console.rule(f"[bold yellow]{spec['label']}  {idx}/{total_records}  {api_target}[/bold yellow]")

    for attempt in range(1, MAX_ATTEMPTS + 1):
        temp = TEMPERATURES[min(attempt - 1, len(TEMPERATURES) - 1)]
        log("try", f"{api_target} attempt {attempt}/{MAX_ATTEMPTS} temp={temp} top_p={TOP_P}", "magenta")
        attempt_debug = {
            "attempt": attempt,
            "messages_before_generation": [dict(m) for m in messages],
            "generation": None,
            "extracted_code": "",
            "validation_issues": [],
            "validation_warnings": [],
            "process": None,
            "status_after_attempt": None,
            "detail": "",
        }

        try:
            raw, repetition, raw_record = stream_llm_response(messages, temperature=temp, max_tokens=MAX_TOKENS)
            attempt_debug["generation"] = raw_record
            total_gen_time += raw_record.get("generation_time_s", 0.0)
            final_token_count = raw_record.get("raw_token_count")
        except (ValueError, RuntimeError) as exc:
            final_status = "CONTEXT_OVERFLOW"
            final_detail = str(exc)
            attempt_debug["status_after_attempt"] = final_status
            attempt_debug["detail"] = final_detail
            attempt_debugs.append(attempt_debug)
            break

        if repetition:
            final_status = "REPETITION_TRAP"
            final_detail = "model repeated lines"
            attempt_debug["status_after_attempt"] = final_status
            attempt_debug["detail"] = final_detail
            attempt_debugs.append(attempt_debug)
            messages = messages[:2]
            messages.append({"role": "user", "content": "Regenerate a shorter non-repeating Python seed program."})
            continue

        code = extract_python_code(raw)
        final_code = code
        issues, warnings = validate_seed_code(code, api_target)
        final_warnings = warnings
        attempt_debug["extracted_code"] = code
        attempt_debug["validation_issues"] = issues
        attempt_debug["validation_warnings"] = warnings

        proc = None
        if not issues:
            try:
                proc, proc_time = run_code_file(code, temp_dir, f"fair_seed_{idx}_{attempt}.py")
                total_proc_time += proc_time
                attempt_debug["process"] = {
                    "returncode": proc.returncode,
                    "stdout": proc.stdout or "",
                    "stderr": proc.stderr or "",
                    "subprocess_time_s": proc_time,
                }
            except subprocess.TimeoutExpired:
                final_status = "TIMEOUT"
                final_detail = f"timeout after {TIMEOUT_S}s"
                attempt_debug["process"] = {"timeout_s": TIMEOUT_S}
                attempt_debug["status_after_attempt"] = final_status
                attempt_debug["detail"] = final_detail
                attempt_debugs.append(attempt_debug)
                messages.append({"role": "assistant", "content": f"```python\n{code}\n```"})
                messages.append({"role": "user", "content": feedback_for_seed(final_status, final_detail, api_target)})
                continue

        final_status, final_detail = evaluate_status(issues, proc)
        attempt_debug["status_after_attempt"] = final_status
        attempt_debug["detail"] = final_detail
        attempt_debugs.append(attempt_debug)

        if final_status == "PASS_RUNNABLE_SEED":
            pass_oneshot = attempt == 1
            log("pass", f"{final_status}: {final_detail[:140]}", "green")
            break

        log("fix", f"{final_status}: {final_detail[:140]}", "yellow")
        messages.append({"role": "assistant", "content": f"```python\n{code}\n```"})
        messages.append({"role": "user", "content": feedback_for_seed(final_status, final_detail, api_target)})

    elapsed = time.time() - t0
    line_count = count_code_lines(final_code)
    summary_detail = {
        "api": api_target,
        "kind": entry["kind"],
        "status": final_status,
        "detail": final_detail,
        "warnings": final_warnings,
        "time_s": round(elapsed, 1),
        "generation_time_s": round(total_gen_time, 3),
        "subprocess_time_s": round(total_proc_time, 3),
        "raw_token_count": final_token_count,
        "code_line_count": line_count,
        "ai_code": final_code,
    }
    debug_detail = {
        **summary_detail,
        "entry": entry,
        "attempts": attempt_debugs,
    }

    console.print(
        f"[bold white]{spec['label']} / {api_target}: {final_status} "
        f"in {elapsed:.1f}s | gen={total_gen_time:.1f}s | tokens={final_token_count} | lines={line_count}[/bold white]"
    )
    if SHOW_CODE_TABLES:
        table = Table(show_header=True, header_style="bold magenta")
        table.add_column("Generated seed", width=90)
        table.add_row(Syntax(final_code or "# no code", "python", theme="monokai", word_wrap=True))
        console.print(table)

    return {
        "status": final_status,
        "pass_oneshot": pass_oneshot,
        "generation_time_s": total_gen_time,
        "subprocess_time_s": total_proc_time,
        "raw_token_count": final_token_count,
        "code_line_count": line_count,
        "warnings": final_warnings,
        "summary_detail": summary_detail,
        "debug_detail": debug_detail,
    }


def run_seed_benchmark(spec, entries):
    results = empty_results(spec)
    debug_details = []

    try:
        with tempfile.TemporaryDirectory() as temp_dir:
            for idx, entry in enumerate(entries, start=1):
                result = run_one_seed_task(spec, entry, idx, len(entries), temp_dir)
                results["total_attempted"] += 1
                status = result["status"]
                if status == "PASS_RUNNABLE_SEED":
                    results["pass_runnable_seed"] += 1
                elif status == "NO_CODE":
                    results["fail_no_code"] += 1
                elif status == "SYNTAX_ERROR":
                    results["fail_syntax"] += 1
                elif status == "TARGET_MISS":
                    results["fail_target_miss"] += 1
                elif status == "FORBIDDEN":
                    results["fail_forbidden"] += 1
                elif status == "RUNTIME_ERROR":
                    results["fail_runtime"] += 1
                elif status == "TIMEOUT":
                    results["fail_timeout"] += 1
                elif status == "REPETITION_TRAP":
                    results["fail_repetition"] += 1
                elif status == "CONTEXT_OVERFLOW":
                    results["fail_context_overflow"] += 1

                if result["pass_oneshot"]:
                    results["pass_oneshot"] += 1
                results["total_generation_time_s"] += result["generation_time_s"]
                results["total_subprocess_time_s"] += result["subprocess_time_s"]
                if result["raw_token_count"] is not None:
                    results["raw_token_counts"].append(result["raw_token_count"])
                results["code_line_counts"].append(result["code_line_count"])
                for warning in result["warnings"]:
                    results["warning_counts"][warning] = results["warning_counts"].get(warning, 0) + 1

                results["details"].append(result["summary_detail"])
                debug_details.append(result["debug_detail"])
    except KeyboardInterrupt:
        results["interrupted"] = True
        console.print(f"\n[bold red]CTRL+C received during {spec['label']}; keeping completed fair benchmark results.[/bold red]")

    return results, debug_details


def mean(values):
    return sum(values) / len(values) if values else 0.0


def print_model_summary(result):
    total = result["total_attempted"] or 1
    t = Table(title=f"FAIR CURRENT SEED BENCHMARK - {result['label']}", style="cyan")
    t.add_column("Metric", style="yellow")
    t.add_column("Count", justify="right")
    t.add_row("Model", f"[bold cyan]{result['model']}[/bold cyan]")
    t.add_row("APIs attempted", str(result["total_attempted"]))
    t.add_row("Runnable target seed", f"[bold green]{result['pass_runnable_seed']}[/bold green]")
    t.add_row("One-shot runnable", f"[green]{result['pass_oneshot']}[/green]")
    t.add_row("-" * 24, "")
    t.add_row("No code", str(result["fail_no_code"]))
    t.add_row("Syntax error", str(result["fail_syntax"]))
    t.add_row("Target miss", str(result["fail_target_miss"]))
    t.add_row("Forbidden construct", str(result["fail_forbidden"]))
    t.add_row("Runtime error", str(result["fail_runtime"]))
    t.add_row("Timeout", str(result["fail_timeout"]))
    t.add_row("Repetition", str(result["fail_repetition"]))
    t.add_row("Context overflow", str(result["fail_context_overflow"]))
    t.add_row("-" * 24, "")
    t.add_row("Runnable rate", f"[bold cyan]{result['pass_runnable_seed']/total*100:.2f}%[/bold cyan]")
    t.add_row("One-shot rate", f"{result['pass_oneshot']/total*100:.2f}%")
    t.add_row("Avg raw tokens", f"{mean(result['raw_token_counts']):.1f}")
    t.add_row("Avg code lines", f"{mean(result['code_line_counts']):.1f}")
    t.add_row("Avg generation sec", f"{result['total_generation_time_s']/total:.2f}")
    console.print(t)


# =============================================================================
# MAIN
# =============================================================================

console.print(Panel(
    "[bold cyan]FAIR CURRENT-RUNTIME PYTORCH SEED BENCHMARK[/bold cyan]\n"
    "[dim]Current torch APIs, OOD-from-train when possible, 40 tasks by default.[/dim]",
    border_style="cyan",
))

entries, dataset_meta = select_eval_apis()
console.print(Panel(json.dumps(dataset_meta, ensure_ascii=False, indent=2), title="Evaluation API Set", border_style="cyan"))

combined = {
    "task": "Fair current-runtime PyTorch seed generation",
    "dataset_meta": dataset_meta,
    "limit_tests": LIMIT_TESTS,
    "max_attempts": MAX_ATTEMPTS,
    "force_cpu_tests": FORCE_CPU_TESTS,
    "sampling": {
        "temperatures": TEMPERATURES,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "min_p": MIN_P,
        "repeat_penalty": REPEAT_PENALTY,
        "max_tokens": MAX_TOKENS,
    },
    "results": {},
}
debug_report = {
    "task": combined["task"],
    "dataset_meta": dataset_meta,
    "models": {},
}

try:
    for spec in MODEL_SPECS:
        load_model(spec)
        result, debug_details = run_seed_benchmark(spec, entries)
        print_model_summary(result)
        combined["results"][spec["label"]] = {k: v for k, v in result.items() if k != "details"}
        combined["results"][spec["label"]]["details"] = result["details"]
        debug_report["models"][spec["label"]] = {
            "summary": {k: v for k, v in result.items() if k != "details"},
            "details": debug_details,
        }
        unload_model()
        if result.get("interrupted"):
            break
except KeyboardInterrupt:
    console.print("\n[bold red]CTRL+C received; writing completed fair benchmark results.[/bold red]")
    unload_model()


compare = Table(title="FAIR CURRENT SEED GENERATOR COMPARISON", style="cyan")
compare.add_column("Metric", style="yellow")
for label in combined["results"].keys():
    compare.add_column(label, justify="right")


def metric(label, key):
    return combined["results"][label].get(key, 0)


for row_label, key in [
    ("APIs attempted", "total_attempted"),
    ("Runnable target seed", "pass_runnable_seed"),
    ("One-shot runnable", "pass_oneshot"),
    ("Syntax error", "fail_syntax"),
    ("Target miss", "fail_target_miss"),
    ("Runtime error", "fail_runtime"),
    ("Timeout", "fail_timeout"),
    ("Context overflow", "fail_context_overflow"),
]:
    compare.add_row(row_label, *[str(metric(label, key)) for label in combined["results"].keys()])

rates = []
one_shot_rates = []
avg_tokens = []
avg_lines = []
avg_gen = []
for label, result in combined["results"].items():
    total = result["total_attempted"] or 1
    rates.append(f"{result['pass_runnable_seed']/total*100:.1f}%")
    one_shot_rates.append(f"{result['pass_oneshot']/total*100:.1f}%")
    avg_tokens.append(f"{mean(result['raw_token_counts']):.1f}")
    avg_lines.append(f"{mean(result['code_line_counts']):.1f}")
    avg_gen.append(f"{result['total_generation_time_s']/total:.2f}s")
compare.add_row("[bold]Runnable rate[/bold]", *[f"[bold cyan]{x}[/bold cyan]" for x in rates])
compare.add_row("[bold]One-shot rate[/bold]", *[f"[bold]{x}[/bold]" for x in one_shot_rates])
compare.add_row("Avg raw tokens", *avg_tokens)
compare.add_row("Avg code lines", *avg_lines)
compare.add_row("Avg gen seconds/API", *avg_gen)
console.print(compare)

with open(REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)
console.print(f"[cyan]Saved fair comparison report: {REPORT_FILE}[/cyan]")

with open(DEBUG_REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(debug_report, f, ensure_ascii=False, indent=2)
console.print(f"[cyan]Saved fair debug report with raw generations: {DEBUG_REPORT_FILE}[/cyan]")


Installing llama-cpp-python CUDA wheel...


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ FAIR CURRENT-RUNTIME PYTORCH SEED BENCHMARK                                                                     │
│ Current torch APIs, OOD-from-train when possible, 40 tasks by default.                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Evaluation API Set ───────────────────────────────────────────────╮
│ {                                                                                                               │
│   "torch_version": "2.10.0+cu128",                                                                              │
│   "torch_cuda": "12.8",                                                                                         │
│   "eval_mode": "current_ood",                                                                                   │
│   "api_pool": "mixed",                                                                                          │
│   "all_current_api_count": 1638,                                                                                │
│   "train_dataset_path": null,                                                                                   │
│   "seen_train_api_count": 0,                                                                                    │
│   "ood_current_api_count": 1638,                                                                                │
│   "selected_api_count": 40,                                                                                     │
│   "selected_bucket_counts": {                                                                                   │
│     "torch": 7,                                                                                                 │
│     "torch.Tensor": 7,                                                                                          │
│     "torch.fft": 7,                                                                                             │
│     "torch.linalg": 7,                                                                                          │
│     "torch.nn.functional": 6,                                                                                   │
│     "torch.special": 6                                                                                          │
│   },                                                                                                            │
│   "selected_apis": [                                                                                            │
│     "torch.LongTensor",                                                                                         │
│     "torch.Tensor.arctan",                                                                                      │
│     "torch.fft.fftn",                                                                                           │
│     "torch.linalg.vector_norm",                                                                                 │
│     "torch.nn.functional.relu_",                                                                                │
│     "torch.special.gammaincc",                                                                                  │
│     "torch.inference_mode",                                                                                     │
│     "torch.Tensor.reciprocal",                                                                                  │
│     "torch.fft.Tensor",                                                                                         │
│     "torch.linalg.matrix_rank",                                                                                 │
│     "torch.nn.functional.embedding",                                                                            │
│     "torch.special.bessel_j0",                                                                                  │
│     "torch.arange",                                                                                             │
│     "torch.Tensor.cfloat",                                                                                      │
│     "torch.fft.irfft2",                               

17:57:16 model Downloading/loading mradermacher/gemma-4-26B-A4B-it-GGUF / gemma-4-26B-A4B-it.Q3_K_M.gguf ...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

base_gemma4_e4b_bf16 / torch.linalg.eigvals: PASS_RUNNABLE_SEED in 20.6s | gen=18.5s | tokens=330 | lines=20

                 FAIR CURRENT SEED BENCHMARK - base_gemma4_e4b_bf16                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                   ┃                                                  Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Model                    │ unsloth/gemma-4-E4B-it-GGUF / gemma-4-E4B-it-BF16.gguf │
│ APIs attempted           │                                                     40 │
│ Runnable target seed     │                                                     40 │
│ One-shot runnable        │                                                     38 │
│ ------------------------ │                                                        │
│ No code                  │                                                      0 │
│ Syntax error             │                                                      0 │
│ Target miss              │                                                      0 │
│ Forbidden construct      │                                                      0 │
│ Runtime error            │                                                      0 │
│ Timeout                  │                                                      0 │
│ Repetition               │                                                      0 │
│ Context overflow         │                                                      0 │
│ ------------------------ │                                                        │
│ Runnable rate            │                                                100.00% │
│ One-shot rate            │                                                 95.00% │
│ Avg raw tokens           │                                                  189.2 │
│ Avg code lines           │                                                   14.8 │
│ Avg generation sec       │                                                  12.39 │
└──────────────────────────┴────────────────────────────────────────────────────────┘

                FAIR CURRENT SEED GENERATOR COMPARISON                
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric               ┃ base_gemma4_26b_q3km ┃ base_gemma4_e4b_bf16 ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ APIs attempted       │                   40 │                   40 │
│ Runnable target seed │                   37 │                   40 │
│ One-shot runnable    │                   31 │                   38 │
│ Syntax error         │                    1 │                    0 │
│ Target miss          │                    1 │                    0 │
│ Runtime error        │                    1 │                    0 │
│ Timeout              │                    0 │                    0 │
│ Context overflow     │                    0 │                    0 │
│ Runnable rate        │                92.5% │               100.0% │
│ One-shot rate        │                77.5% │                95.0% │
│ Avg raw tokens       │                341.8 │                189.2 │
│ Avg code lines       │                 21.2 │                 14.8 │
│ Avg gen seconds/API  │               14.31s │               12.39s │
└──────────────────────┴──────────────────────┴──────────────────────┘

Saved fair comparison report: benchmark_seed_generator_base26_vs_e4b_bf16.json

Saved fair debug report with raw generations: benchmark_seed_generator_base26_vs_e4b_bf16_debug.json